# Digital Banking App Analytics — Exploratory Data Analysis

This notebook explores a synthetic product analytics dataset for a Vietnamese digital banking app.

The analysis focuses on four areas:

1. Data quality and table relationships
2. Onboarding funnel performance
3. Acquisition channel quality
4. Retention and churn risk signals

The dataset is synthetic and assumption-based. It is designed for portfolio demonstration, not as real banking customer data.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

print("Current folder:", Path.cwd())
print("Data folder exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    print("Files in data folder:")
    for file in DATA_DIR.iterdir():
        print("-", file.name)

Current folder: C:\Users\ninav\Desktop\digital-banking-app-analytics
Data folder exists: True
Files in data folder:
- app_events.csv
- churn_labels.csv
- transactions.csv
- users.csv


In [2]:
users = pd.read_csv("data/users.csv", parse_dates=["registration_date"])
events = pd.read_csv("data/app_events.csv", parse_dates=["event_time"])
transactions = pd.read_csv("data/transactions.csv", parse_dates=["txn_date"])
churn = pd.read_csv("data/churn_labels.csv")

print("users:", users.shape)
print("events:", events.shape)
print("transactions:", transactions.shape)
print("churn:", churn.shape)

users: (5000, 7)
events: (27927, 5)
transactions: (10523, 5)
churn: (5000, 4)


In [3]:
print("USERS columns:")
print(users.columns.tolist())
display(users.head())

print("EVENTS columns:")
print(events.columns.tolist())
display(events.head())

print("TRANSACTIONS columns:")
print(transactions.columns.tolist())
display(transactions.head())

print("CHURN columns:")
print(churn.columns.tolist())
display(churn.head())

USERS columns:
['user_id', 'age_group', 'city_tier', 'income_band', 'acquisition_channel', 'device_os', 'registration_date']


,user_id,age_group,city_tier,income_band,acquisition_channel,device_os,registration_date
0,U00001,25-34,Tier 1,Mid,Paid Ads,iOS,2024-03-22
1,U00002,55+,Tier 2,Mid,Paid Ads,Android,2024-01-15
2,U00003,35-44,Tier 3,Low,Paid Ads,Android,2024-01-04
3,U00004,25-34,Tier 1,Mid,Organic,Android,2024-04-04
4,U00005,18-24,Tier 3,Mid,Organic,Android,2024-02-05


EVENTS columns:
['user_id', 'event_time', 'event_name', 'session_id', 'status']


,user_id,event_time,event_name,session_id,status
0,U00001,2024-03-22 01:19:00,app_install,SU00001_app_,success
1,U00001,2024-03-22 06:08:00,register_start,SU00001_regi,success
2,U00001,2024-03-22 14:16:00,register_complete,SU00001_regi,success
3,U00001,2024-03-22 18:18:00,ekyc_start,SU00001_ekyc,success
4,U00001,2024-03-22 22:30:00,ekyc_complete,SU00001_ekyc,success


TRANSACTIONS columns:
['user_id', 'txn_date', 'txn_type', 'amount_vnd', 'status']


,user_id,txn_date,txn_type,amount_vnd,status
0,U00007,2024-02-03 19:00:00,savings_deposit,655000,success
1,U00007,2024-02-20 13:00:00,card_payment,338000,success
2,U00007,2024-03-16 14:00:00,transfer,1882000,success
3,U00009,2024-04-07 10:00:00,card_payment,3607000,success
4,U00009,2024-04-13 19:00:00,loan_repayment,9551000,success


CHURN columns:
['user_id', 'is_active_d30', 'is_active_d60', 'churn_flag']


,user_id,is_active_d30,is_active_d60,churn_flag
0,U00001,False,False,True
1,U00002,False,False,True
2,U00003,False,False,True
3,U00004,False,False,True
4,U00005,False,False,True


In [4]:
tables = {
    "users": users,
    "events": events,
    "transactions": transactions,
    "churn": churn
}

for name, df in tables.items():
    print("=" * 60)
    print(name.upper())
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    
    print("\nMissing values:")
    print(df.isna().sum())
    
    print("\nData types:")
    print(df.dtypes)

USERS
Shape: (5000, 7)
Duplicate rows: 0

Missing values:
user_id                0
age_group              0
city_tier              0
income_band            0
acquisition_channel    0
device_os              0
registration_date      0
dtype: int64

Data types:
user_id                           str
age_group                         str
city_tier                         str
income_band                       str
acquisition_channel               str
device_os                         str
registration_date      datetime64[us]
dtype: object
EVENTS
Shape: (27927, 5)
Duplicate rows: 0

Missing values:
user_id       0
event_time    0
event_name    0
session_id    0
status        0
dtype: int64

Data types:
user_id                  str
event_time    datetime64[us]
event_name               str
session_id               str
status                   str
dtype: object
TRANSACTIONS
Shape: (10523, 5)
Duplicate rows: 0

Missing values:
user_id       0
txn_date      0
txn_type      0
amount_vnd    0
status

In [5]:
print("Unique users in users:", users["user_id"].nunique())
print("Unique users in events:", events["user_id"].nunique())
print("Unique users in transactions:", transactions["user_id"].nunique())
print("Unique users in churn:", churn["user_id"].nunique())

print("\nUser IDs in events but not in users:")
print(len(set(events["user_id"]) - set(users["user_id"])))

print("\nUser IDs in transactions but not in users:")
print(len(set(transactions["user_id"]) - set(users["user_id"])))

print("\nUser IDs in churn but not in users:")
print(len(set(churn["user_id"]) - set(users["user_id"])))

Unique users in users: 5000
Unique users in events: 5000
Unique users in transactions: 1478
Unique users in churn: 5000

User IDs in events but not in users:
0

User IDs in transactions but not in users:
0

User IDs in churn but not in users:
0


In [6]:
activation = (
    events[
        (events["event_name"] == "account_activated") &
        (events["status"] == "success")
    ]
    .groupby("user_id", as_index=False)
    .agg(activation_date=("event_time", "min"))
)

print("Activated users:", activation["user_id"].nunique())
display(activation.head())

Activated users: 1725


,user_id,activation_date
0,U00001,2024-03-23 05:18:00
1,U00007,2024-01-29 23:06:00
2,U00009,2024-04-05 05:13:00
3,U00012,2024-04-05 03:15:00
4,U00013,2024-04-25 23:09:00


In [7]:
first_txn = (
    transactions[transactions["status"] == "success"]
    .groupby("user_id", as_index=False)
    .agg(first_txn_date=("txn_date", "min"))
)

print("Users with successful transaction:", first_txn["user_id"].nunique())
display(first_txn.head())

Users with successful transaction: 1465


,user_id,first_txn_date
0,U00007,2024-02-03 19:00:00
1,U00009,2024-04-07 10:00:00
2,U00012,2024-04-05 15:00:00
3,U00013,2024-04-30 14:00:00
4,U00017,2024-03-05 10:00:00


In [8]:
txn_summary = (
    transactions
    .groupby("user_id", as_index=False)
    .agg(
        total_txns=("txn_date", "count"),
        successful_txns=("status", lambda x: (x == "success").sum()),
        failed_txns=("status", lambda x: (x == "failed").sum()),
        total_amount_vnd=("amount_vnd", "sum"),
        avg_amount_vnd=("amount_vnd", "mean")
    )
)

print("Users with any transaction:", txn_summary["user_id"].nunique())
display(txn_summary.head())

Users with any transaction: 1478


,user_id,total_txns,successful_txns,failed_txns,total_amount_vnd,avg_amount_vnd
0,U00007,3,3,0,2875000,9.583333e+05
1,U00009,8,8,0,35902000,4.487750e+06
2,U00012,19,19,0,109651000,5.771105e+06
3,U00013,6,6,0,250945000,4.182417e+07
4,U00015,1,0,1,5869000,5.869000e+06


In [9]:
feature_events = [
    "transfer_initiated",
    "qr_payment",
    "bill_payment",
    "savings_opened",
    "card_payment",
    "loan_viewed"
]

feature_usage = (
    events[
        (events["event_name"].isin(feature_events)) &
        (events["status"] == "success")
    ]
    .pivot_table(
        index="user_id",
        columns="event_name",
        values="event_time",
        aggfunc="min"
    )
    .notna()
    .astype(int)
    .reset_index()
)

feature_usage["n_features"] = feature_usage[feature_events].sum(axis=1)

feature_usage["has_bill_or_savings"] = (
    (feature_usage["bill_payment"] == 1) |
    (feature_usage["savings_opened"] == 1)
).astype(int)

print("Users with at least one feature event:", feature_usage["user_id"].nunique())
display(feature_usage.head())

Users with at least one feature event: 1550


event_name,user_id,bill_payment,card_payment,loan_viewed,qr_payment,savings_opened,transfer_initiated,n_features,has_bill_or_savings
0,U00009,0,1,1,0,0,0,2,0
1,U00012,1,1,0,1,1,1,5,1
2,U00013,1,0,0,0,0,1,2,1
3,U00017,0,0,1,0,0,0,1,0
4,U00021,0,1,0,0,0,0,1,0


In [10]:
# Remove pivot column name from feature_usage display
feature_usage.columns.name = None

user_analytics = (
    users
    .merge(activation, on="user_id", how="left")
    .merge(first_txn, on="user_id", how="left")
    .merge(txn_summary, on="user_id", how="left")
    .merge(feature_usage, on="user_id", how="left")
    .merge(churn, on="user_id", how="left")
)

user_analytics["activated"] = user_analytics["activation_date"].notna().astype(int)

user_analytics["days_to_first_txn"] = (
    user_analytics["first_txn_date"] - user_analytics["activation_date"]
).dt.days

print("user_analytics shape:", user_analytics.shape)
display(user_analytics.head())

user_analytics shape: (5000, 27)


,user_id,age_group,city_tier,income_band,acquisition_channel,device_os,registration_date,activation_date,first_txn_date,total_txns,...,qr_payment,savings_opened,transfer_initiated,n_features,has_bill_or_savings,is_active_d30,is_active_d60,churn_flag,activated,days_to_first_txn
0,U00001,25-34,Tier 1,Mid,Paid Ads,iOS,2024-03-22,2024-03-23 05:18:00,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,1,NaN
1,U00002,55+,Tier 2,Mid,Paid Ads,Android,2024-01-15,NaT,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,0,NaN
2,U00003,35-44,Tier 3,Low,Paid Ads,Android,2024-01-04,NaT,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,0,NaN
3,U00004,25-34,Tier 1,Mid,Organic,Android,2024-04-04,NaT,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,0,NaN
4,U00005,18-24,Tier 3,Mid,Organic,Android,2024-02-05,NaT,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,True,0,NaN


In [11]:
num_cols = [
    "total_txns",
    "successful_txns",
    "failed_txns",
    "total_amount_vnd",
    "avg_amount_vnd"
]

for col in num_cols:
    user_analytics[col] = user_analytics[col].fillna(0)

feature_cols = [
    "bill_payment",
    "card_payment",
    "loan_viewed",
    "qr_payment",
    "savings_opened",
    "transfer_initiated",
    "n_features",
    "has_bill_or_savings"
]

for col in feature_cols:
    user_analytics[col] = user_analytics[col].fillna(0).astype(int)

print(user_analytics[num_cols + feature_cols].isna().sum())
display(user_analytics.head())

total_txns             0
successful_txns        0
failed_txns            0
total_amount_vnd       0
avg_amount_vnd         0
bill_payment           0
card_payment           0
loan_viewed            0
qr_payment             0
savings_opened         0
transfer_initiated     0
n_features             0
has_bill_or_savings    0
dtype: int64


,user_id,age_group,city_tier,income_band,acquisition_channel,device_os,registration_date,activation_date,first_txn_date,total_txns,...,qr_payment,savings_opened,transfer_initiated,n_features,has_bill_or_savings,is_active_d30,is_active_d60,churn_flag,activated,days_to_first_txn
0,U00001,25-34,Tier 1,Mid,Paid Ads,iOS,2024-03-22,2024-03-23 05:18:00,NaT,0.0,...,0,0,0,0,0,False,False,True,1,NaN
1,U00002,55+,Tier 2,Mid,Paid Ads,Android,2024-01-15,NaT,NaT,0.0,...,0,0,0,0,0,False,False,True,0,NaN
2,U00003,35-44,Tier 3,Low,Paid Ads,Android,2024-01-04,NaT,NaT,0.0,...,0,0,0,0,0,False,False,True,0,NaN
3,U00004,25-34,Tier 1,Mid,Organic,Android,2024-04-04,NaT,NaT,0.0,...,0,0,0,0,0,False,False,True,0,NaN
4,U00005,18-24,Tier 3,Mid,Organic,Android,2024-02-05,NaT,NaT,0.0,...,0,0,0,0,0,False,False,True,0,NaN


In [14]:
# First transaction timing bucket
user_analytics["first_txn_bucket"] = "No transaction"

user_analytics.loc[
    user_analytics["days_to_first_txn"] <= 7,
    "first_txn_bucket"
] = "Within 7 days"

user_analytics.loc[
    user_analytics["days_to_first_txn"] > 7,
    "first_txn_bucket"
] = "After 7 days"


# Feature adoption bucket
user_analytics["feature_bucket"] = "0 features"

user_analytics.loc[
    user_analytics["n_features"] == 1,
    "feature_bucket"
] = "1 feature"

user_analytics.loc[
    user_analytics["n_features"] >= 2,
    "feature_bucket"
] = "2+ features"


# Failed transaction bucket
user_analytics["failed_txn_bucket"] = "0 failed"

user_analytics.loc[
    (user_analytics["failed_txns"] >= 1) & (user_analytics["failed_txns"] <= 2),
    "failed_txn_bucket"
] = "1-2 failed"

user_analytics.loc[
    user_analytics["failed_txns"] >= 3,
    "failed_txn_bucket"
] = "3+ failed"


# Registration month
user_analytics["registration_month"] = (
    user_analytics["registration_date"]
    .dt.to_period("M")
    .astype(str)
)

display(
    user_analytics[
        [
            "user_id",
            "activated",
            "days_to_first_txn",
            "first_txn_bucket",
            "n_features",
            "feature_bucket",
            "failed_txns",
            "failed_txn_bucket",
            "registration_month",
            "churn_flag"
        ]
    ].head(10)
)

,user_id,activated,days_to_first_txn,first_txn_bucket,n_features,feature_bucket,failed_txns,failed_txn_bucket,registration_month,churn_flag
0,U00001,1,NaN,No transaction,0,0 features,0.0,0 failed,2024-03,True
1,U00002,0,NaN,No transaction,0,0 features,0.0,0 failed,2024-01,True
2,U00003,0,NaN,No transaction,0,0 features,0.0,0 failed,2024-01,True
3,U00004,0,NaN,No transaction,0,0 features,0.0,0 failed,2024-04,True
4,U00005,0,NaN,No transaction,0,0 features,0.0,0 failed,2024-02,True
5,U00006,0,NaN,No transaction,0,0 features,0.0,0 failed,2024-02,True
6,U00007,1,4.0,Within 7 days,0,0 features,0.0,0 failed,2024-01,False
7,U00008,0,NaN,No transaction,0,0 features,0.0,0 failed,2024-01,True
8,U00009,1,2.0,Within 7 days,2,2+ features,0.0,0 failed,2024-04,True
9,U00010,0,NaN,No transaction,0,0 features,0.0,0 failed,2024-01,True


In [15]:
total_users = users["user_id"].nunique()
activated_users = user_analytics["activated"].sum()
activation_rate = activated_users / total_users

d30_active_users = user_analytics["is_active_d30"].sum()
d30_retention_rate = d30_active_users / total_users

d60_active_users = user_analytics["is_active_d60"].sum()
d60_retention_rate = d60_active_users / total_users

churned_users = user_analytics["churn_flag"].sum()
churn_rate = churned_users / total_users

total_transactions = len(transactions)
successful_transactions = (transactions["status"] == "success").sum()
failed_transactions = (transactions["status"] == "failed").sum()
transaction_success_rate = successful_transactions / total_transactions

avg_txns_per_activated_user = total_transactions / activated_users

kpi_summary = pd.DataFrame({
    "metric": [
        "Total Users",
        "Activated Users",
        "Activation Rate",
        "D30 Active Users",
        "D30 Retention Rate",
        "D60 Active Users",
        "D60 Retention Rate",
        "Churned Users",
        "Churn Rate",
        "Total Transactions",
        "Transaction Success Rate",
        "Avg Transactions per Activated User"
    ],
    "value": [
        total_users,
        activated_users,
        activation_rate,
        d30_active_users,
        d30_retention_rate,
        d60_active_users,
        d60_retention_rate,
        churned_users,
        churn_rate,
        total_transactions,
        transaction_success_rate,
        avg_txns_per_activated_user
    ]
})

display(kpi_summary)

,metric,value
0,Total Users,5000.000000
1,Activated Users,1725.000000
2,Activation Rate,0.345000
3,D30 Active Users,1229.000000
4,D30 Retention Rate,0.245800
5,D60 Active Users,747.000000
6,D60 Retention Rate,0.149400
7,Churned Users,4253.000000
8,Churn Rate,0.850600
9,Total Transactions,10523.000000


In [16]:
funnel_steps = [
    "app_install",
    "register_start",
    "register_complete",
    "ekyc_start",
    "ekyc_complete",
    "account_activated"
]

funnel = (
    events[
        (events["event_name"].isin(funnel_steps)) &
        (events["status"] == "success")
    ]
    .groupby("event_name", as_index=False)
    .agg(users_reached=("user_id", "nunique"))
)

funnel["step_order"] = funnel["event_name"].map({
    step: i + 1 for i, step in enumerate(funnel_steps)
})

funnel = funnel.sort_values("step_order")

install_users = funnel.loc[
    funnel["event_name"] == "app_install",
    "users_reached"
].iloc[0]

funnel["conversion_from_install"] = funnel["users_reached"] / install_users
funnel["step_to_step_conversion"] = funnel["users_reached"] / funnel["users_reached"].shift(1)

display(funnel)

,event_name,users_reached,step_order,conversion_from_install,step_to_step_conversion
1,app_install,5000,1,1.0000,NaN
5,register_start,4617,2,0.9234,0.923400
4,register_complete,3886,3,0.7772,0.841672
3,ekyc_start,3467,4,0.6934,0.892177
2,ekyc_complete,2300,5,0.4600,0.663398
0,account_activated,1725,6,0.3450,0.750000


In [17]:
channel_summary = (
    user_analytics
    .groupby("acquisition_channel", as_index=False)
    .agg(
        users=("user_id", "nunique"),
        activated_users=("activated", "sum"),
        d30_retention=("is_active_d30", "mean"),
        d60_retention=("is_active_d60", "mean"),
        churn_rate=("churn_flag", "mean"),
        avg_txns_per_user=("total_txns", "mean")
    )
)

channel_summary["activation_rate"] = (
    channel_summary["activated_users"] / channel_summary["users"]
)

channel_summary = channel_summary.sort_values("activation_rate", ascending=False)

display(channel_summary)

,acquisition_channel,users,activated_users,d30_retention,d60_retention,churn_rate,avg_txns_per_user,activation_rate
4,Referral,974,434,0.339836,0.223819,0.776181,2.662218,0.445585
1,Organic,1197,480,0.314954,0.206349,0.793651,2.420217,0.401003
0,Branch,512,199,0.277344,0.152344,0.847656,2.380859,0.388672
3,Partnership,747,256,0.235609,0.140562,0.859438,2.100402,0.342704
2,Paid Ads,1570,356,0.129299,0.063057,0.936943,1.429936,0.226752


In [18]:
first_txn_summary = (
    user_analytics[user_analytics["activated"] == 1]
    .groupby("first_txn_bucket", as_index=False)
    .agg(
        users=("user_id", "nunique"),
        d30_retention=("is_active_d30", "mean"),
        d60_retention=("is_active_d60", "mean"),
        churn_rate=("churn_flag", "mean")
    )
    .sort_values("d30_retention", ascending=False)
)

feature_churn = (
    user_analytics[user_analytics["activated"] == 1]
    .groupby("feature_bucket", as_index=False)
    .agg(
        users=("user_id", "nunique"),
        d30_retention=("is_active_d30", "mean"),
        d60_retention=("is_active_d60", "mean"),
        churn_rate=("churn_flag", "mean")
    )
    .sort_values("churn_rate", ascending=False)
)

failed_txn_summary = (
    user_analytics[user_analytics["activated"] == 1]
    .groupby("failed_txn_bucket", as_index=False)
    .agg(
        users=("user_id", "nunique"),
        d30_retention=("is_active_d30", "mean"),
        d60_retention=("is_active_d60", "mean"),
        churn_rate=("churn_flag", "mean")
    )
    .sort_values("churn_rate", ascending=False)
)

print("First transaction timing summary:")
display(first_txn_summary)

print("Feature adoption vs churn:")
display(feature_churn)

print("Failed transactions vs churn:")
display(failed_txn_summary)

First transaction timing summary:


,first_txn_bucket,users,d30_retention,d60_retention,churn_rate
2,Within 7 days,781,0.781050,0.500640,0.499360
0,After 7 days,684,0.738304,0.450292,0.549708
1,No transaction,260,0.438462,0.184615,0.815385


Feature adoption vs churn:


,feature_bucket,users,d30_retention,d60_retention,churn_rate
0,0 features,175,0.382857,0.131429,0.868571
1,1 feature,428,0.630841,0.329439,0.670561
2,2+ features,1122,0.795009,0.519608,0.480392


Failed transactions vs churn:


,failed_txn_bucket,users,d30_retention,d60_retention,churn_rate
2,3+ failed,45,0.688889,0.266667,0.733333
0,0 failed,1087,0.686293,0.411224,0.588776
1,1-2 failed,593,0.762226,0.485666,0.514334


In [19]:
cohort_summary = (
    user_analytics
    .groupby("registration_month", as_index=False)
    .agg(
        users=("user_id", "nunique"),
        activated_users=("activated", "sum"),
        d30_retention=("is_active_d30", "mean"),
        d60_retention=("is_active_d60", "mean"),
        churn_rate=("churn_flag", "mean")
    )
)

cohort_summary["activation_rate"] = (
    cohort_summary["activated_users"] / cohort_summary["users"]
)

display(cohort_summary)

,registration_month,users,activated_users,d30_retention,d60_retention,churn_rate,activation_rate
0,2024-01,1269,442,0.249803,0.159180,0.840820,0.348306
1,2024-02,1287,441,0.240093,0.142968,0.857032,0.342657
2,2024-03,1362,478,0.245228,0.146109,0.853891,0.350954
3,2024-04,1082,364,0.248614,0.149723,0.850277,0.336414


In [20]:
from pathlib import Path

EXPORT_DIR = Path("data/processed")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

user_analytics.to_csv(EXPORT_DIR / "user_analytics.csv", index=False)
kpi_summary.to_csv(EXPORT_DIR / "kpi_summary.csv", index=False)
funnel.to_csv(EXPORT_DIR / "funnel_summary.csv", index=False)
channel_summary.to_csv(EXPORT_DIR / "channel_summary.csv", index=False)
first_txn_summary.to_csv(EXPORT_DIR / "first_txn_summary.csv", index=False)
feature_churn.to_csv(EXPORT_DIR / "feature_churn.csv", index=False)
failed_txn_summary.to_csv(EXPORT_DIR / "failed_txn_summary.csv", index=False)
cohort_summary.to_csv(EXPORT_DIR / "cohort_summary.csv", index=False)

print("Exported files:")
for file in EXPORT_DIR.iterdir():
    print("-", file.name)

Exported files:
- channel_summary.csv
- cohort_summary.csv
- failed_txn_summary.csv
- feature_churn.csv
- first_txn_summary.csv
- funnel_summary.csv
- kpi_summary.csv
- user_analytics.csv
